# NLP Lab: Dynamic N-Gram Sentence Generator

This notebook builds on the original N-gram backoff generator and integrates the full
Lab 1 + Lab 2 toolkit into one executable pipeline, while satisfying the teacher's
additional requirements.

**Covered in this notebook:**

- Text preprocessing: regex cleaning, tokenization
- Stop word removal, Porter stemming, WordNet lemmatization
- Text representation: Bag of Words and TF-IDF
- Edit distance (Levenshtein) and corpus-based spelling correction
- **Dictionary-based** spelling correction beyond the corpus vocabulary (new)
- Sentence boundary markers (`<s>`, `</s>`)
- Dynamic N-gram count tables and MLE probability tables (any order, no hardcoding)
- **Laplace (Add-One) smoothing**, generalized for any N (new)
- Backoff for unseen histories
- **Non-deterministic**, weighted-probability next-word prediction (Shannon's-game style) (new)
- **Automatic N-gram order detection** from the number of words the user types (new)
- Full autoregressive sentence generation


In [1]:
import re
import math
import random
import sys
import nltk
import numpy as np

from collections import defaultdict, Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

for resource, package in [("tokenizers/punkt", "punkt"),
                          ("tokenizers/punkt_tab", "punkt_tab"),
                          ("corpora/stopwords", "stopwords"),
                          ("corpora/wordnet", "wordnet"),
                          ("corpora/omw-1.4", "omw-1.4")]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Corpus


In [2]:
with open("corpus.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

print(raw_text[:500])

Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do.
So she was considering in her own mind whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies.
The river went on flowing quietly, and the afternoon light lay warm across the grass.
Birds moved through the trees, and the air was full of soft summer sounds.

In the middle of the field stood an old stone cottage with a sloping roof and a nar


## 2. Maximum Supported N-Gram Order

The teacher no longer wants the user to type `N` manually (Requirement 1). Instead,
`N` (the n-gram order actually used for a prediction) is **detected automatically**
later from the number of words the user types as their sentence-so-far (Section 12).

Because the corpus must be padded with `N-1` sentence-start markers (`<s>`) *before*
we know the user's `N`, we pad using a fixed **upper bound**, `MAX_N` — the largest
n-gram order the notebook is prepared to build backoff tables for. If a user types
more than `MAX_N` words, only the last `MAX_N` are used.


In [3]:
# Largest n-gram order the notebook builds a backoff/count table for.
# The n-gram order actually used for a given query is detected automatically
# from the number of words the user types (Section 12) and can be anywhere
# from 1 (unigram) up to MAX_N.
MAX_N = 6

print(f"Maximum supported n-gram order: {MAX_N}")

Maximum supported n-gram order: 6


## 3. Sentence Splitting and Preprocessing (for the Language Model)


In [4]:
def preprocess_text(text, n):

    sentences = sent_tokenize(text)

    processed_sentences = []

    for sentence in sentences:

        sentence = re.sub(r'\[[^]]*\]', '', sentence)
        sentence = re.sub(r'--|\.\.\.', '', sentence)
        sentence = re.sub(r'\d+', '', sentence)

        start_markers = " ".join(["<s>"] * (n - 1))
        sentence = start_markers + " " + sentence + " </s>"

        sentence = re.sub(r"[^a-zA-Z'<>\s/]", "", sentence)
        sentence = re.sub(r'\s+', ' ', sentence).strip()

        tokens = sentence.lower().split()

        processed_sentences.append(tokens)

    return processed_sentences


sentences = preprocess_text(raw_text, MAX_N)

print(sentences[:3])

[['<s>', '<s>', '<s>', '<s>', '<s>', 'alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on', 'the', 'bank', 'and', 'of', 'having', 'nothing', 'to', 'do', '</s>'], ['<s>', '<s>', '<s>', '<s>', '<s>', 'so', 'she', 'was', 'considering', 'in', 'her', 'own', 'mind', 'whether', 'the', 'pleasure', 'of', 'making', 'a', 'daisychain', 'would', 'be', 'worth', 'the', 'trouble', 'of', 'getting', 'up', 'and', 'picking', 'the', 'daisies', '</s>'], ['<s>', '<s>', '<s>', '<s>', '<s>', 'the', 'river', 'went', 'on', 'flowing', 'quietly', 'and', 'the', 'afternoon', 'light', 'lay', 'warm', 'across', 'the', 'grass', '</s>']]


In [5]:
tokens = []

for sentence in sentences:
    tokens.extend(sentence)

vocabulary = set(tokens)

# Vocabulary used for NEXT-WORD prediction/sampling should never include the
# sentence-start marker -- a real generated word should never be "<s>".
predictable_vocab = sorted(vocabulary - {"<s>"})

print("Total tokens:", len(tokens))
print("Vocabulary size:", len(vocabulary))
print("Predictable vocabulary size:", len(predictable_vocab))

Total tokens: 1452
Vocabulary size: 564
Predictable vocabulary size: 563


## 4. Stop Word Removal, Stemming & Lemmatization

These normalization steps (from Lab 1) are demonstrated here on a raw corpus sentence.

Note: the n-gram generator itself (Sections 8 onward) deliberately keeps stop words
and does **not** stem/lemmatize its training tokens — a language model needs the exact
surface word forms and function words (`the`, `is`, `to`, ...) to predict fluent,
grammatical continuations. Stemming/lemmatizing would collapse different word forms
together and break the sentence-generation quality. The tools are still fully
implemented and demonstrated below, as required.


In [6]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def full_preprocess(text, remove_stopwords=True, use_lemmatization=True):
    """Lab 1 style preprocessing pipeline: lowercase, regex clean, tokenize,
    optional stop word removal, optional lemmatization."""
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    words = word_tokenize(text)

    if remove_stopwords:
        words = [w for w in words if w not in stop_words]

    if use_lemmatization:
        words = [lemmatizer.lemmatize(w) for w in words]

    return words


sample_sentence = sent_tokenize(raw_text)[1]
print("Original sentence:", sample_sentence)

tokens_before = word_tokenize(re.sub(r'[^a-zA-Z\s]', '', sample_sentence.lower()))
print("\nTokens (before stop word removal):", tokens_before)

tokens_after_stopwords = [w for w in tokens_before if w not in stop_words]
print("Tokens (after stop word removal): ", tokens_after_stopwords)

stemmed_words = [stemmer.stem(w) for w in tokens_after_stopwords]
lemmatized_words = [lemmatizer.lemmatize(w) for w in tokens_after_stopwords]

print("\nStemming vs Lemmatization")
for original, stem, lemma in zip(tokens_after_stopwords, stemmed_words, lemmatized_words):
    print(f"  {original:<15} stem: {stem:<15} lemma: {lemma}")

Original sentence: So she was considering in her own mind whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies.

Tokens (before stop word removal): ['so', 'she', 'was', 'considering', 'in', 'her', 'own', 'mind', 'whether', 'the', 'pleasure', 'of', 'making', 'a', 'daisychain', 'would', 'be', 'worth', 'the', 'trouble', 'of', 'getting', 'up', 'and', 'picking', 'the', 'daisies']
Tokens (after stop word removal):  ['considering', 'mind', 'whether', 'pleasure', 'making', 'daisychain', 'would', 'worth', 'trouble', 'getting', 'picking', 'daisies']

Stemming vs Lemmatization
  considering     stem: consid          lemma: considering
  mind            stem: mind            lemma: mind
  whether         stem: whether         lemma: whether
  pleasure        stem: pleasur         lemma: pleasure
  making          stem: make            lemma: making
  daisychain      stem: daisychain      lemma: daisychain
  would           stem: would       

## 5. Text Representation: Bag of Words & TF-IDF

Bag of Words (BoW) represents each document as raw word-frequency counts, discarding
grammar and order. TF-IDF additionally down-weights words that are common across many
documents:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D), \qquad
\text{TF}(t, d) = \frac{C(t, d)}{\sum_{t' \in d} C(t', d)}, \qquad
\text{IDF}(t, D) = \log\!\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

A handful of corpus sentences (treated as separate "documents") are used to
demonstrate both representations below.


In [7]:
demo_documents = [
    " ".join(full_preprocess(sentence, remove_stopwords=True, use_lemmatization=True))
    for sentence in sent_tokenize(raw_text)[:6]
]

print("--- 1. Bag of Words Representation ---")
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(demo_documents)

print("Vocabulary Found :", bow_vectorizer.get_feature_names_out())
print("BoW Matrix Array :\n", bow_matrix.toarray())

print("\n--- 2. TF-IDF Representation ---")
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(demo_documents)

print("TF-IDF Matrix Array:\n", np.round(tfidf_matrix.toarray(), 3))

--- 1. Bag of Words Representation ---
Vocabulary Found : ['across' 'afternoon' 'air' 'alice' 'bank' 'beginning' 'bell' 'beside'
 'bird' 'brass' 'brick' 'considering' 'cottage' 'daisy' 'daisychain'
 'door' 'field' 'flowing' 'frame' 'full' 'gate' 'get' 'getting' 'grass'
 'hung' 'lay' 'led' 'light' 'making' 'middle' 'mind' 'moved' 'narrow'
 'nothing' 'old' 'path' 'picking' 'pleasure' 'porch' 'quietly' 'river'
 'roof' 'sister' 'sitting' 'sloping' 'soft' 'sound' 'stone' 'stood'
 'summer' 'tired' 'tree' 'trouble' 'warm' 'went' 'whether' 'worn' 'worth'
 'would']
BoW Matrix Array :
 [[0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0
  0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0
  1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1]
 [1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0
  0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0]
 [0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 

## 6. Levenshtein Distance (Corpus-Only Correction)


In [8]:
def calculate_edit_distance(word1, word2):

    m, n = len(word1), len(word2)

    dp = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[m][n]


def correct_word_edit_distance(word, vocab):
    """Corpus-only spelling correction (original approach): finds the
    closest word that already exists in the training vocabulary."""

    if word in vocab:
        return word

    distances = {}

    for valid_word in vocab:

        if valid_word == "<s>" or valid_word == "</s>":
            continue

        distance = calculate_edit_distance(word, valid_word)
        distances[valid_word] = distance

    best_word = min(distances, key=distances.get)

    return best_word

## 7. Dictionary-Based Spell Checking (New)

The corpus-only corrector above can only fix a typo if the *correct* spelling already
happens to appear in `corpus.txt`. That fails for a perfectly ordinary English word the
small corpus never used (e.g. `langauge -> language`, `machne -> machine`).

[`pyspellchecker`](https://pypi.org/project/pyspellchecker/) ships with a large,
general-English word-frequency dictionary, so it can correct such typos even when the
correct word never appears in the corpus.

**Correction strategy** (best of both worlds):
1. If the word is already in the corpus vocabulary, keep it unchanged.
2. Otherwise, ask `pyspellchecker` for its best general-English dictionary correction.
3. If the dictionary has no suggestion at all, fall back to the corpus edit-distance
   corrector from Section 6 so the word still maps to *some* known token.


In [10]:
import sys
import subprocess

try:
    from spellchecker import SpellChecker
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspellchecker", "-q"])
    from spellchecker import SpellChecker

spell = SpellChecker()


def correct_word(word, vocab, spell_checker=spell):
    """Correct `word` using the corpus vocabulary first, then a general
    English dictionary, falling back to corpus edit-distance if the
    dictionary has no suggestion."""

    word = word.lower()

    if word in vocab:
        return word

    dictionary_correction = spell_checker.correction(word)

    if dictionary_correction is None:
        corrected = correct_word_edit_distance(word, vocab)
        print(f"Corrected '{word}' to '{corrected}' (corpus edit-distance)")
        return corrected

    if dictionary_correction != word:
        print(f"Corrected '{word}' to '{dictionary_correction}' (dictionary)")

    return dictionary_correction

## 8. Dynamic N-Gram Counting (Any Order)

Rather than writing separate code for unigram / bigram / trigram counting (or
hardcoding a single fixed `N`), one generalized function builds the count table for
*any* history length. It is reused for every order from `0` (unigram, empty history)
up to `MAX_N - 1`, which is what lets Section 11's backoff work for any `N`.


In [11]:
def build_ngram_counts(sentence_list, history_length):
    """Generalized n-gram counter.

    history_length = 0 -> unigram counts (history is the empty tuple)
    history_length = 1 -> bigram counts
    history_length = k -> (k+1)-gram counts
    """
    count_table = defaultdict(Counter)

    for sentence in sentence_list:
        for i in range(history_length, len(sentence)):

            history = tuple(sentence[i - history_length:i])
            next_word = sentence[i]

            if next_word == "<s>":
                continue

            count_table[history][next_word] += 1

    return count_table


count_tables = {
    history_length: build_ngram_counts(sentences, history_length)
    for history_length in range(MAX_N)
}

print(f"Dynamic n-gram count tables created for orders 1 through {MAX_N}.")

Dynamic n-gram count tables created for orders 1 through 6.


## 9. Laplace (Add-One) Smoothing

For a history `h` of length `N-1` and a candidate next word `w`, the unsmoothed
maximum-likelihood estimate is

$$P(w \mid h) = \frac{C(h, w)}{C(h)}$$

which assigns **zero** probability to any word that was never observed after `h`.
Laplace smoothing fixes this by adding 1 to every count and adding the vocabulary size
`V` to the denominator, so every word in the vocabulary keeps a small, non-zero
probability:

$$P_{Laplace}(w \mid h) = \frac{C(h, w) + 1}{C(h) + V}$$

This generalizes to *any* n-gram order because `count_tables[history_length]` already
stores `C(h, w)` for that specific order — only `V` (size of the predictable
vocabulary) is shared across every order.


In [12]:
VOCAB_SIZE = len(predictable_vocab)


def laplace_probability(word, history, count_table, vocab_size=VOCAB_SIZE, k=1):
    """Add-k (Laplace when k=1) smoothed probability of `word` following `history`."""
    counts = count_table.get(history, Counter())
    total = sum(counts.values())
    return (counts[word] + k) / (total + k * vocab_size)


def get_smoothed_distribution(history, count_table, vocab, k=1):
    """Full Laplace-smoothed probability distribution over `vocab` for a history."""
    counts = count_table.get(history, Counter())
    total = sum(counts.values())
    v = len(vocab)
    return {word: (counts[word] + k) / (total + k * v) for word in vocab}


# Demonstration: smoothed bigram probability table for one example history
example_history = ("i",)

if example_history in count_tables[1]:
    example_distribution = get_smoothed_distribution(
        example_history, count_tables[1], predictable_vocab
    )
    top_words = Counter(example_distribution).most_common(10)
    print(f"Smoothed P(w | {example_history}):")
    for word, prob in top_words:
        print(f"  {word:<12} {prob:.5f}")
else:
    print(f"'{example_history}' was not seen as a bigram history in this corpus.")

'('i',)' was not seen as a bigram history in this corpus.


## 10. Non-Deterministic Next-Word Prediction (Shannon's-Game Style)

The original notebook always picked `argmax P(w | h)`, so the same input produced the
same output every time. Instead, the teacher wants **weighted random sampling**: words
with higher probability should be picked more often, but lower-probability words should
still have a chance of appearing.

`random.choices(population, weights=...)` implements exactly that. `sample_next_word`
takes an `rng` object (defaulting to the plain `random` module) so a seed is entirely
**optional** — pass a seeded `random.Random(seed)` for reproducible output, or leave
the default for a different result on every run.

`shannons_predict` wraps this into a single-step "next word" demonstration, similar to
Claude Shannon's classic guessing game: it shows the top candidate words with their
smoothed probabilities, then samples one.


In [13]:
def sample_next_word(distribution, rng=random):
    """Weighted random sample of the next word from a probability distribution."""
    words = list(distribution.keys())
    weights = list(distribution.values())
    return rng.choices(words, weights=weights, k=1)[0]


def shannons_predict(history_words, count_tables, vocab, k=1, top_n=5):
    """Single next-word prediction step: find the best available (backoff)
    history, show its top smoothed candidates, then sample one word."""

    max_history_length = min(len(history_words), MAX_N - 1)

    chosen_history_length, chosen_history = None, None

    for history_length in range(max_history_length, -1, -1):
        history = tuple(history_words[-history_length:]) if history_length else ()

        if history in count_tables[history_length] and count_tables[history_length][history]:
            chosen_history_length, chosen_history = history_length, history
            break

    if chosen_history_length is None:
        print("No continuation was found for this history.")
        return None

    distribution = get_smoothed_distribution(
        chosen_history, count_tables[chosen_history_length], vocab, k
    )

    print(f"History used: {chosen_history} (order {chosen_history_length + 1} model)")
    print("Top candidates:")
    for word, prob in Counter(distribution).most_common(top_n):
        print(f"  {word:<12} {prob:.4f}")

    predicted_word = sample_next_word(distribution)
    print(f"Sampled next word: '{predicted_word}'")

    return predicted_word


# Demonstration -- run this a few times, the sampled word can change each time
_ = shannons_predict(["i", "love"], count_tables, predictable_vocab)

History used: () (order 1 model)
Top candidates:
  the          0.0603
  </s>         0.0412
  and          0.0352
  a            0.0293
  of           0.0131
Sampled next word: 'pinned'


## 11. Automatic N Detection + User Input

The user simply types their sentence so far. The n-gram order `N` is set to the number
of words they typed (Requirement 1) — there is no prompt asking for `N`. Each word is
then spell-corrected with the dictionary-aware corrector from Section 7.

| Words typed | N (order) used |
|---|---|
| `I` | 1 (unigram) |
| `I love` | 2 (bigram) |
| `I love natural` | 3 (trigram) |
| `I love natural language` | 4 (4-gram) |

If more than `MAX_N` words are typed, only the last `MAX_N` are kept.


In [14]:
raw_input_text = input("Enter your sentence so far: ").strip()

while not raw_input_text:
    print("Please type at least one word.")
    raw_input_text = input("Enter your sentence so far: ").strip()

typed_words = raw_input_text.lower().split()

if len(typed_words) > MAX_N:
    print(f"Only the last {MAX_N} words will be used (MAX_N = {MAX_N}).")
    typed_words = typed_words[-MAX_N:]

N = len(typed_words)  # <-- automatically detected, never asked for
print(f"Detected n-gram order: N = {N}")

corrected_words = [correct_word(word, vocabulary) for word in typed_words]

# Full context is kept for display; the generation loop below only ever
# looks back at the last (N-1) tokens of it when building n-gram history,
# exactly matching an order-N model.
generated_words = corrected_words

print("Starting context:", generated_words)

Detected n-gram order: N = 5
Starting context: ['alice', 'is', 'a', 'good', 'boy']


## 12. Sentence Generation (Backoff + Smoothing + Sampling)

Starting from `generated_words`, the notebook repeatedly:

1. Tries the highest order first: the last `N-1` words as history.
2. If that exact history was never observed, backs off to a shorter history
   (`N-2`, `N-3`, ... down to the empty/unigram history, which always exists).
3. Builds the Laplace-smoothed distribution over the whole predictable vocabulary for
   the chosen history (Section 9).
4. Samples the next word from that distribution (Section 10) instead of always taking
   the single most probable word.

Generation stops at `</s>` or after `MAX_GENERATED_WORDS` words.


In [15]:
MAX_GENERATED_WORDS = 50
GENERATION_SEED = None  # set an int for reproducible generation, or leave None

# A single rng is created for the whole generation so an optional seed
# reproduces the entire sentence, not just a single word.
rng = random.Random(GENERATION_SEED) if GENERATION_SEED is not None else random

reached_end_marker = False

for _ in range(MAX_GENERATED_WORDS):

    chosen_history_length = None
    chosen_history = None

    for history_length in range(N - 1, -1, -1):

        history = tuple(generated_words[-history_length:]) if history_length else ()

        if history in count_tables[history_length] and count_tables[history_length][history]:
            chosen_history_length = history_length
            chosen_history = history
            break

    if chosen_history_length is None:
        print("No continuation was found; stopping generation.")
        break

    distribution = get_smoothed_distribution(
        chosen_history, count_tables[chosen_history_length], predictable_vocab
    )

    next_word = sample_next_word(distribution, rng=rng)

    generated_words.append(next_word)

    if next_word == "</s>":
        reached_end_marker = True
        break

if not reached_end_marker:
    print(f"Stopped after {MAX_GENERATED_WORDS} words to avoid an infinite loop.")


output_words = [word for word in generated_words if word not in ("<s>", "</s>")]
generated_sentence = " ".join(output_words)

print("\nGenerated Sentence:")
print(generated_sentence)

Stopped after 50 words to avoid an infinite loop.

Generated Sentence:
alice is a good boy newer window continued symbols buses forward glowed clues tired kinds bell light even process collection doors the carrying smell darkened air mannequin collecting are process looked splitting often many shadows hung history easy wide picking curved sheets conductor spend clues conversations drew symbols cracked shone blank belong kneaded shapes get
